# Day 3: Time Series Analysis & Feature Engineering

## Smart City IoT Analytics Pipeline

### 🎯 LEARNING OBJECTIVES
- Perform time series analysis on sensor data
- Calculate correlations between different sensor types
- Engineer features for predictive modeling
- Implement window functions for trend analysis

### 📅 SCHEDULE
Morning (4 hours):
1. Temporal Pattern Analysis (2 hours)
2. Cross-Sensor Correlation Analysis (2 hours)

Afternoon (4 hours):
3. Feature Engineering (3 hours)
4. Trend Analysis (1 hour)

### ✅ DELIVERABLES
- Time series analysis dashboard
- Correlation study findings
- Feature engineering pipeline
- Trend analysis reports

## IMPORTS AND SETUP

In [9]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

# PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Machine learning imports
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.stat import Correlation
from pyspark.ml.linalg import Vectors

# Initialize Spark Session
try:
    spark.sparkContext.setLogLevel("WARN")
    print("✅ Using existing Spark session")
except NameError:
    spark = (
        SparkSession.builder
        .appName("SmartCityIoTPipeline-Day3")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.ui.enabled", "false")
        .config("spark.eventLog.enabled", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")
    print("✅ Created new Spark session")

print("📈 Day 3: Time Series Analysis & Feature Engineering")
print("=" * 60)

✅ Using existing Spark session
📈 Day 3: Time Series Analysis & Feature Engineering


## LOAD CLEANED DATA FROM DAY 2

In [10]:
print("\n📂 Loading cleaned data from Day 2...")

def load_cleaned_datasets():
    """
    Load cleaned datasets created during Day 2.

    Zones are loaded from the raw CSV because Day 2 did not create
    a cleaned zones dataset.
    """
    datasets = {}

    processed_data_dir = "../data/processed"
    raw_data_dir = "../data/raw"

    try:
        # Zones were not part of the Day 2 cleaned output.
        datasets["zones"] = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(f"{raw_data_dir}/city_zones.csv")
        )

        # Day 2 cleaned datasets were saved as Parquet.
        cleaned_mappings = {
            "traffic": f"{processed_data_dir}/traffic_cleaned.parquet",
            "air_quality": f"{processed_data_dir}/air_quality_cleaned.parquet",
            "weather": f"{processed_data_dir}/weather_cleaned.parquet",
            "energy": f"{processed_data_dir}/energy_cleaned.parquet",
            "occupancy": f"{processed_data_dir}/occupancy_cleaned.parquet",
            "fiscal": f"{processed_data_dir}/fiscal_cleaned.parquet",
        }

        missing_paths = [
            path for path in cleaned_mappings.values()
            if not os.path.exists(path)
        ]

        if missing_paths:
            print("❌ One or more Day 2 cleaned datasets are missing:")
            for path in missing_paths:
                print(f"   - {path}")
            print("Run the Day 2 save section before continuing to Day 3.")
            return {}

        for name, file_path in cleaned_mappings.items():
            datasets[name] = spark.read.parquet(file_path)

        print("✅ Day 2 cleaned datasets loaded successfully")
        return datasets

    except Exception as e:
        print(f"❌ Error loading cleaned datasets: {str(e)}")
        return {}

datasets = load_cleaned_datasets()

# Quick data overview
print("\n📊 Dataset Overview:")
for name, df in datasets.items():
    if df is not None:
        count = df.count()
        print(f"   📊 {name}: {count:,} records")


📂 Loading cleaned data from Day 2...
✅ Day 2 cleaned datasets loaded successfully

📊 Dataset Overview:
   📊 zones: 36,000 records
   📊 traffic: 36,000 records
   📊 air_quality: 36,000 records
   📊 weather: 36,000 records
   📊 energy: 36,000 records
   📊 occupancy: 36,000 records
   📊 fiscal: 36,000 records


## SECTION 1: TEMPORAL PATTERN ANALYSIS (Morning - 2 hours)

In [11]:
print("\n" + "=" * 60)
print("⏰ SECTION 1: TEMPORAL PATTERN ANALYSIS")
print("=" * 60)


⏰ SECTION 1: TEMPORAL PATTERN ANALYSIS


### TODO 1.1: Seasonal Decomposition Analysis (60 minutes)

**🎯 TASK:** Decompose time series into trend, seasonal, and residual components  
**💡 HINT:** Look for daily, weekly, and monthly patterns  
**📚 CONCEPTS:** Seasonality, trends, cyclical patterns, decomposition

In [12]:
def analyze_temporal_patterns(df, value_col, time_col="timestamp", sensor_col=None):
    """
    Analyze temporal patterns in sensor data.

    Args:
        df: DataFrame with time series data
        value_col: Column containing values to analyze
        time_col: Timestamp column
        sensor_col: Sensor ID column (optional)

    Returns:
        DataFrame with temporal pattern analysis
    """
    print(f"\n📈 Temporal Pattern Analysis: {value_col}")
    print("-" * 40)

    # TODO: Add time-based features for pattern analysis
    df_with_time = (
        df.withColumn("year", F.year(time_col))
          .withColumn("month", F.month(time_col))
          .withColumn("day", F.dayofmonth(time_col))
          .withColumn("hour", F.hour(time_col))
          .withColumn("day_of_week", F.dayofweek(time_col))
          .withColumn("week_of_year", F.weekofyear(time_col))
          .withColumn(
              "is_weekend",
              F.when(F.dayofweek(time_col).isin([1, 7]), True).otherwise(False)
          )
    )

    # TODO: Hourly patterns
    print("🕐 Hourly Patterns:")
    hourly_patterns = (
        df_with_time.groupBy("hour")
        .agg(
            F.avg(value_col).alias("avg_value"),
            F.stddev(value_col).alias("stddev_value"),
            F.min(value_col).alias("min_value"),
            F.max(value_col).alias("max_value"),
            F.count(value_col).alias("count_readings")
        )
        .orderBy("hour")
    )
    hourly_patterns.show(24)

    # TODO: Find peak and off-peak hours
    peak_hours = hourly_patterns.orderBy(F.desc("avg_value")).limit(3)
    print("   🔝 Peak hours:")
    peak_hours.show()

    # TODO: Day of week patterns
    print("\n📅 Day of Week Patterns:")
    daily_patterns = (
        df_with_time.groupBy("day_of_week")
        .agg(
            F.avg(value_col).alias("avg_value"),
            F.count(value_col).alias("count_readings")
        )
        .orderBy("day_of_week")
    )

    daily_patterns = daily_patterns.withColumn(
        "day_name",
        F.when(F.col("day_of_week") == 1, "Sunday")
         .when(F.col("day_of_week") == 2, "Monday")
         .when(F.col("day_of_week") == 3, "Tuesday")
         .when(F.col("day_of_week") == 4, "Wednesday")
         .when(F.col("day_of_week") == 5, "Thursday")
         .when(F.col("day_of_week") == 6, "Friday")
         .when(F.col("day_of_week") == 7, "Saturday")
    )

    daily_patterns.select("day_name", "avg_value", "count_readings").show()

    # TODO: Weekend vs Weekday comparison
    weekend_vs_weekday = df_with_time.groupBy("is_weekend").agg(
        F.avg(value_col).alias("avg_value"),
        F.count(value_col).alias("count_readings")
    )

    print("\n🏖️ Weekend vs Weekday:")
    weekend_vs_weekday.show()

    # TODO: Monthly patterns (seasonal trends)
    print("\n📊 Monthly Patterns:")
    monthly_patterns = (
        df_with_time.groupBy("month")
        .agg(
            F.avg(value_col).alias("avg_value"),
            F.count(value_col).alias("count_readings")
        )
        .orderBy("month")
    )

    monthly_patterns.show(12)

    return df_with_time


# TODO: Analyze temporal patterns for each sensor type
temporal_results = {}

# Traffic patterns
if "traffic" in datasets and datasets["traffic"] is not None:
    print("🚗 TRAFFIC TEMPORAL ANALYSIS")
    print("=" * 40)

    traffic_temporal = analyze_temporal_patterns(
        datasets["traffic"],
        "vehicle_count"
    )
    temporal_results["traffic_vehicle_count"] = traffic_temporal

    if "avg_speed" in datasets["traffic"].columns:
        speed_temporal = analyze_temporal_patterns(
            datasets["traffic"],
            "avg_speed"
        )
        temporal_results["traffic_speed"] = speed_temporal

# Air quality patterns
if "air_quality" in datasets and datasets["air_quality"] is not None:
    print("\n🌫️ AIR QUALITY TEMPORAL ANALYSIS")
    print("=" * 40)

    if "pm25" in datasets["air_quality"].columns:
        air_temporal = analyze_temporal_patterns(
            datasets["air_quality"],
            "pm25"
        )
        temporal_results["air_quality_pm25"] = air_temporal

# Weather patterns
if "weather" in datasets and datasets["weather"] is not None:
    print("\n🌤️ WEATHER TEMPORAL ANALYSIS")
    print("=" * 40)

    if "temperature" in datasets["weather"].columns:
        weather_temporal = analyze_temporal_patterns(
            datasets["weather"],
            "temperature"
        )
        temporal_results["weather_temperature"] = weather_temporal

# Energy consumption patterns
if "energy" in datasets and datasets["energy"] is not None:
    print("\n⚡ ENERGY TEMPORAL ANALYSIS")
    print("=" * 40)

    if "power_consumption" in datasets["energy"].columns:
        energy_temporal = analyze_temporal_patterns(
            datasets["energy"],
            "power_consumption"
        )
        temporal_results["energy_power"] = energy_temporal

# Hotel occupancy patterns
if "occupancy" in datasets and datasets["occupancy"] is not None:
    print("\n🏨 OCCUPANCY TEMPORAL ANALYSIS")
    print("=" * 40)

    if "occupied_rooms" in datasets["occupancy"].columns:
        occupancy_temporal = analyze_temporal_patterns(
            datasets["occupancy"],
            "occupied_rooms"
        )
        temporal_results["occupancy_rate"] = occupancy_temporal

# Fiscal patterns
if "fiscal" in datasets and datasets["fiscal"] is not None:
    print("\n💰 FISCAL TEMPORAL ANALYSIS")
    print("=" * 40)

    if "expense" in datasets["fiscal"].columns:
        expense_temporal = analyze_temporal_patterns(
            datasets["fiscal"],
            "expense"
        )
        temporal_results["expense_rate"] = expense_temporal

    if "revenue" in datasets["fiscal"].columns:
        revenue_temporal = analyze_temporal_patterns(
            datasets["fiscal"],
            "revenue"
        )
        temporal_results["revenue_rate"] = revenue_temporal

🚗 TRAFFIC TEMPORAL ANALYSIS

📈 Temporal Pattern Analysis: vehicle_count
----------------------------------------
🕐 Hourly Patterns:
+----+------------------+------------------+---------+---------+--------------+
|hour|         avg_value|      stddev_value|min_value|max_value|count_readings|
+----+------------------+------------------+---------+---------+--------------+
|   0| 69.52133333333333|28.634713946258973|       18|      120|          1500|
|   1| 68.15933333333334|29.509430422801145|       15|      120|          1500|
|   2| 68.19825268817205|28.818712111379412|       15|      120|          1488|
|   3| 68.20833333333333| 29.14891789235353|       15|      120|          1512|
|   4|             68.82|28.889929364384386|       15|      120|          1500|
|   5| 66.65733333333333|28.723009134223812|       17|      120|          1500|
|   6| 68.08333333333333| 28.83460904320836|       15|      120|          1500|
|   7|104.25333333333333|45.164939241776544|       23|      186|    

### TODO 1.2: Pattern Anomaly Detection (60 minutes)

**🎯 TASK:** Identify deviations from expected temporal patterns  
**💡 HINT:** Compare actual patterns with historical averages  
**📚 CONCEPTS:** Anomaly detection, pattern deviation, threshold setting

In [13]:
def detect_pattern_anomalies(df, value_col, time_col="timestamp"):
    """
    Detect anomalies in temporal patterns.

    Args:
        df: DataFrame with temporal features
        value_col: Value column to analyze
        time_col: Timestamp column

    Returns:
        DataFrame with anomaly flags
    """
    print(f"\n🚨 Pattern Anomaly Detection: {value_col}")
    print("-" * 35)

    # TODO: Calculate expected values based on historical patterns
    # Use moving averages and seasonal patterns.
    daily_window = (
        Window.partitionBy("hour")
        .orderBy(time_col)
        .rowsBetween(-7, 7)
    )

    weekly_window = (
        Window.partitionBy("day_of_week", "hour")
        .orderBy(time_col)
        .rowsBetween(-4, 4)
    )

    df_with_expected = (
        df.withColumn(
            f"{value_col}_expected_daily",
            F.avg(value_col).over(daily_window)
        )
        .withColumn(
            f"{value_col}_expected_weekly",
            F.avg(value_col).over(weekly_window)
        )
    )

    # TODO: Calculate deviations from expected patterns
    df_with_deviations = (
        df_with_expected.withColumn(
            f"{value_col}_deviation_daily",
            F.abs(
                F.col(value_col) -
                F.col(f"{value_col}_expected_daily")
            )
        )
        .withColumn(
            f"{value_col}_deviation_weekly",
            F.abs(
                F.col(value_col) -
                F.col(f"{value_col}_expected_weekly")
            )
        )
    )

    # TODO: Calculate deviation thresholds
    daily_std = (
        df_with_deviations
        .agg(
            F.stddev(
                f"{value_col}_deviation_daily"
            ).alias("daily_std")
        )
        .first()["daily_std"]
    )

    weekly_std = (
        df_with_deviations
        .agg(
            F.stddev(
                f"{value_col}_deviation_weekly"
            ).alias("weekly_std")
        )
        .first()["weekly_std"]
    )

    # TODO: Flag anomalies
    if daily_std is not None and weekly_std is not None:
        df_with_anomalies = (
            df_with_deviations.withColumn(
                f"{value_col}_anomaly_daily",
                F.when(
                    F.col(f"{value_col}_deviation_daily") > 2 * daily_std,
                    True
                ).otherwise(False)
            )
            .withColumn(
                f"{value_col}_anomaly_weekly",
                F.when(
                    F.col(f"{value_col}_deviation_weekly") > 2 * weekly_std,
                    True
                ).otherwise(False)
            )
        )

        # TODO: Count anomalies
        daily_anomalies = df_with_anomalies.filter(
            F.col(f"{value_col}_anomaly_daily") == True
        ).count()

        weekly_anomalies = df_with_anomalies.filter(
            F.col(f"{value_col}_anomaly_weekly") == True
        ).count()

        print(f"   📊 Daily pattern anomalies: {daily_anomalies}")
        print(f"   📊 Weekly pattern anomalies: {weekly_anomalies}")

        return df_with_anomalies

    return df_with_deviations


# TODO: Detect pattern anomalies in traffic data
if "traffic_vehicle_count" in temporal_results:
    traffic_with_anomalies = detect_pattern_anomalies(
        temporal_results["traffic_vehicle_count"],
        "vehicle_count"
    )

    if "vehicle_count_anomaly_daily" in traffic_with_anomalies.columns:
        print("\n🚨 Sample Daily Anomalies in Traffic:")
        anomalies = (
            traffic_with_anomalies
            .filter(F.col("vehicle_count_anomaly_daily") == True)
            .select(
                "timestamp",
                "vehicle_count",
                "vehicle_count_expected_daily",
                "hour",
                "day_of_week"
            )
        )
        anomalies.show(10)


🚨 Pattern Anomaly Detection: vehicle_count
-----------------------------------
   📊 Daily pattern anomalies: 11165
   📊 Weekly pattern anomalies: 10475

🚨 Sample Daily Anomalies in Traffic:
+-------------------+-------------+----------------------------+----+-----------+
|          timestamp|vehicle_count|vehicle_count_expected_daily|hour|day_of_week|
+-------------------+-------------+----------------------------+----+-----------+
|2025-01-01 00:00:00|          109|                      71.375|   0|          4|
|2025-01-02 00:05:00|           24|                        64.8|   0|          5|
|2025-01-02 00:20:00|           21|           69.06666666666666|   0|          5|
|2025-01-02 00:25:00|          105|           64.66666666666667|   0|          5|
|2025-01-02 00:35:00|           22|          60.666666666666664|   0|          5|
|2025-01-02 00:45:00|          116|                        62.0|   0|          5|
|2025-01-03 00:00:00|           22|           61.53333333333333|   0|  

## SECTION 2: CROSS-SENSOR CORRELATION ANALYSIS

### TODO 2.1: Multi-Sensor Correlation Matrix

**Task:** Calculate correlations between different sensor types.  
**Hint:** Join datasets on timestamp and location for meaningful correlations.  
**Concepts:** Correlation analysis, data fusion, causal relationships.


In [14]:
print("\n" + "=" * 60)
print("🔗 SECTION 2: CROSS-SENSOR CORRELATION ANALYSIS")
print("=" * 60)

def prepare_correlation_dataset(datasets):
    """Prepare an hourly combined dataset for cross-sensor correlation analysis."""
    print("\n🔄 Preparing combined dataset for correlation analysis...")

    if "traffic" not in datasets or datasets["traffic"] is None:
        print("❌ Traffic data not available for correlation analysis")
        return None

    traffic_hourly = (
        datasets["traffic"]
        .withColumn("hour_timestamp", F.date_trunc("hour", "timestamp"))
        .groupBy("hour_timestamp")
        .agg(
            F.avg("vehicle_count").alias("avg_vehicle_count"),
            F.avg("avg_speed").alias("avg_traffic_speed"),
            F.count("*").alias("traffic_readings")
        )
    )
    combined_df = traffic_hourly

    if "air_quality" in datasets and datasets["air_quality"] is not None:
        air_hourly = (
            datasets["air_quality"]
            .withColumn("hour_timestamp", F.date_trunc("hour", "timestamp"))
            .groupBy("hour_timestamp")
            .agg(
                F.avg("pm25").alias("avg_pm25"),
                F.avg("no2").alias("avg_no2"),
                F.avg("temperature").alias("avg_air_temp"),
                F.count("*").alias("air_readings")
            )
        )
        combined_df = combined_df.join(air_hourly, "hour_timestamp", "outer")

    if "weather" in datasets and datasets["weather"] is not None:
        weather_aggs = [
            F.avg("temperature").alias("avg_weather_temp"),
            F.avg("humidity").alias("avg_humidity"),
            F.avg("wind_speed").alias("avg_wind_speed"),
            F.count("*").alias("weather_readings"),
        ]
        if "precipitation" in datasets["weather"].columns:
            weather_aggs.insert(3, F.avg("precipitation").alias("avg_precipitation"))

        weather_hourly = (
            datasets["weather"]
            .withColumn("hour_timestamp", F.date_trunc("hour", "timestamp"))
            .groupBy("hour_timestamp")
            .agg(*weather_aggs)
        )
        combined_df = combined_df.join(weather_hourly, "hour_timestamp", "outer")

    if "energy" in datasets and datasets["energy"] is not None:
        energy_hourly = (
            datasets["energy"]
            .withColumn("hour_timestamp", F.date_trunc("hour", "timestamp"))
            .groupBy("hour_timestamp")
            .agg(
                F.avg("power_consumption").alias("avg_power_consumption"),
                F.count("*").alias("energy_readings")
            )
        )
        combined_df = combined_df.join(energy_hourly, "hour_timestamp", "outer")

    if "occupancy" in datasets and datasets["occupancy"] is not None:
        occupancy_df = datasets["occupancy"]
        occupancy_hourly = (
            occupancy_df
            .withColumn(
                "occupancy_rate",
                F.when(
                    (F.col("available_rooms") + F.col("occupied_rooms")) > 0,
                    F.col("occupied_rooms") /
                    (F.col("available_rooms") + F.col("occupied_rooms"))
                )
            )
            .withColumn("hour_timestamp", F.date_trunc("hour", "timestamp"))
            .groupBy("hour_timestamp")
            .agg(
                F.avg("occupancy_rate").alias("avg_occupancy_rate"),
                F.avg("guests").alias("avg_guests"),
                F.count("*").alias("occupancy_readings")
            )
        )
        combined_df = combined_df.join(occupancy_hourly, "hour_timestamp", "outer")

    if "fiscal" in datasets and datasets["fiscal"] is not None:
        fiscal_hourly = (
            datasets["fiscal"]
            .withColumn("hour_timestamp", F.date_trunc("hour", "timestamp"))
            .groupBy("hour_timestamp")
            .agg(
                F.avg("expense").alias("avg_expense"),
                F.avg("revenue").alias("avg_revenue"),
                F.count("*").alias("fiscal_readings")
            )
        )
        combined_df = combined_df.join(fiscal_hourly, "hour_timestamp", "outer")

    combined_df = (
        combined_df
        .withColumn("hour", F.hour("hour_timestamp"))
        .withColumn("day_of_week", F.dayofweek("hour_timestamp"))
        .withColumn(
            "is_weekend",
            F.when(F.dayofweek("hour_timestamp").isin([1, 7]), True).otherwise(False)
        )
    )

    print(f"✅ Combined dataset created with {combined_df.count():,} hourly records")
    return combined_df


combined_data = prepare_correlation_dataset(datasets)

if combined_data is not None:
    print("\n📊 Combined Dataset Overview:")
    combined_data.printSchema()


🔗 SECTION 2: CROSS-SENSOR CORRELATION ANALYSIS

🔄 Preparing combined dataset for correlation analysis...


✅ Combined dataset created with 17,998 hourly records

📊 Combined Dataset Overview:
root
 |-- hour_timestamp: timestamp (nullable = true)
 |-- avg_vehicle_count: double (nullable = true)
 |-- avg_traffic_speed: double (nullable = true)
 |-- traffic_readings: long (nullable = true)
 |-- avg_pm25: double (nullable = true)
 |-- avg_no2: double (nullable = true)
 |-- avg_air_temp: double (nullable = true)
 |-- air_readings: long (nullable = true)
 |-- avg_weather_temp: double (nullable = true)
 |-- avg_humidity: double (nullable = true)
 |-- avg_wind_speed: double (nullable = true)
 |-- avg_precipitation: double (nullable = true)
 |-- weather_readings: long (nullable = true)
 |-- avg_power_consumption: double (nullable = true)
 |-- energy_readings: long (nullable = true)
 |-- avg_occupancy_rate: double (nullable = true)
 |-- avg_guests: double (nullable = true)
 |-- occupancy_readings: long (nullable = true)
 |-- avg_expense: double (nullable = true)
 |-- avg_revenue: double (nullable = tr

### TODO 2.1 continued: Calculate Correlation Matrix

In [16]:
# TODO 2.1 continued: Calculate Correlation Matrix

import builtins

def calculate_sensor_correlations(df):
    """
    Calculate Pearson correlations between available sensor measurements.
    """

    print("\n🧮 Calculating Cross-Sensor Correlations")
    print("-" * 40)

    numeric_cols = [
        "avg_vehicle_count",
        "avg_traffic_speed",
        "avg_pm25",
        "avg_no2",
        "avg_air_temp",
        "avg_weather_temp",
        "avg_humidity",
        "avg_wind_speed",
        "avg_precipitation",
        "avg_power_consumption",
        "avg_occupancy_rate",
        "avg_guests",
        "avg_expense",
        "avg_revenue"
    ]

    # Only use columns that actually exist
    available_cols = [
        col for col in numeric_cols
        if col in df.columns
    ]

    print(
        f"📋 Analyzing correlations for: "
        f"{available_cols}"
    )

    if len(available_cols) < 2:
        print("❌ Not enough numeric columns for correlation analysis")
        return None

    correlation_results = {}

    print("\n📊 Correlation Results:")
    print("=" * 60)

    # Compare each column with every other column once
    for i in range(len(available_cols)):

        col1 = available_cols[i]

        for j in range(i + 1, len(available_cols)):

            col2 = available_cols[j]

            # Use only rows where both values exist
            pair_df = (
                df
                .select(col1, col2)
                .na.drop()
            )

            record_count = pair_df.count()

            if record_count > 1:

                corr_val = pair_df.stat.corr(
                    col1,
                    col2,
                    method="pearson"
                )

                correlation_results[
                    f"{col1}_vs_{col2}"
                ] = corr_val

                print(
                    f"{col1} vs {col2}: "
                    f"{corr_val:.3f} "
                    f"({record_count:,} records)"
                )

    # ---------------------------------------------------------
    # Strongest correlations
    # ---------------------------------------------------------

    print("\n🔝 Strongest Correlations:")

    valid_correlations = [
        (pair, corr)
        for pair, corr in correlation_results.items()
        if corr is not None and not np.isnan(corr)
    ]

    sorted_correlations = sorted(
        valid_correlations,
        key=lambda x: builtins.abs(x[1]),
        reverse=True
    )

    for pair, corr in sorted_correlations[:5]:

        strength = (
            "Strong"
            if builtins.abs(corr) > 0.7
            else "Moderate"
            if builtins.abs(corr) > 0.5
            else "Weak"
        )

        print(
            f"   {pair}: "
            f"{corr:.3f} ({strength})"
        )

    return correlation_results


correlations = (
    calculate_sensor_correlations(combined_data)
    if combined_data is not None
    else None
)


🧮 Calculating Cross-Sensor Correlations
----------------------------------------
📋 Analyzing correlations for: ['avg_vehicle_count', 'avg_traffic_speed', 'avg_pm25', 'avg_no2', 'avg_air_temp', 'avg_weather_temp', 'avg_humidity', 'avg_wind_speed', 'avg_precipitation', 'avg_power_consumption', 'avg_occupancy_rate', 'avg_guests', 'avg_expense', 'avg_revenue']

📊 Correlation Results:
avg_vehicle_count vs avg_traffic_speed: -0.586 (2,999 records)
avg_vehicle_count vs avg_pm25: -0.015 (2,999 records)
avg_vehicle_count vs avg_no2: -0.025 (2,999 records)
avg_vehicle_count vs avg_air_temp: -0.038 (2,999 records)
avg_vehicle_count vs avg_weather_temp: -0.001 (2,999 records)
avg_vehicle_count vs avg_humidity: -0.004 (2,999 records)
avg_vehicle_count vs avg_wind_speed: -0.002 (2,999 records)
avg_vehicle_count vs avg_precipitation: -0.027 (2,999 records)
avg_vehicle_count vs avg_power_consumption: 0.295 (2,999 records)
avg_vehicle_count vs avg_occupancy_rate: 0.010 (2,999 records)
avg_vehicle_coun

### TODO 2.2: Spatial Correlation Analysis

**Task:** Analyze correlations between nearby sensors.  
**Hint:** Sensors close to each other should show similar patterns.  
**Concepts:** Spatial correlation, distance calculations, geographic clustering.


In [24]:
# TODO 2.2: Spatial Correlation Analysis

print("\n" + "=" * 60)
print("🗺️ SPATIAL CORRELATION ANALYSIS")
print("=" * 60)

import math
import builtins


def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate distance between two latitude/longitude points in kilometers.
    """
    earth_radius_km = 6371.0

    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)

    delta_lat = math.radians(lat2 - lat1)
    delta_lon = math.radians(lon2 - lon1)

    a = (
        math.sin(delta_lat / 2) ** 2
        + math.cos(lat1_rad)
        * math.cos(lat2_rad)
        * math.sin(delta_lon / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a)
    )

    return earth_radius_km * c


def analyze_spatial_correlations(
    df,
    value_col,
    sensor_col="sensor_id",
    lat_col="latitude",
    lon_col="longitude",
    max_distance_km=2.0,
    sample_size=100
):
    """
    Analyze whether nearby sensors show similar hour-of-day patterns.

    Because individual sensors in this dataset have sparse readings and
    do not report at matching timestamps, correlation is calculated using
    average values by hour of day rather than exact timestamps.
    """

    print(f"\n🗺️ Spatial Correlation Analysis: {value_col}")
    print("-" * 40)

    # ---------------------------------------------------------
    # Get sensor locations
    # ---------------------------------------------------------

    sensor_locations = (
        df
        .select(
            sensor_col,
            lat_col,
            lon_col
        )
        .dropDuplicates([sensor_col])
        .limit(sample_size)
        .collect()
    )

    print(
        f"📍 Analyzing a sample of {len(sensor_locations)} sensors "
        "(from the full sensor population)"
    )

    # ---------------------------------------------------------
    # Find nearby sensor pairs
    # ---------------------------------------------------------

    nearby_pairs = []

    for i in range(len(sensor_locations)):

        sensor1 = sensor_locations[i]

        for j in range(i + 1, len(sensor_locations)):

            sensor2 = sensor_locations[j]

            distance = haversine_distance(
                sensor1[lat_col],
                sensor1[lon_col],
                sensor2[lat_col],
                sensor2[lon_col]
            )

            if distance <= max_distance_km:

                nearby_pairs.append(
                    (
                        sensor1[sensor_col],
                        sensor2[sensor_col],
                        distance
                    )
                )

    print(
        f"🔍 Found {len(nearby_pairs)} sampled sensor pairs "
        f"within {max_distance_km} km"
    )

    # ---------------------------------------------------------
    # Create hour-of-day patterns for each sensor
    # ---------------------------------------------------------

    hourly_patterns = (
        df
        .withColumn(
            "hour_of_day",
            F.hour("timestamp")
        )
        .groupBy(
            sensor_col,
            "hour_of_day"
        )
        .agg(
            F.avg(value_col).alias("hourly_avg_value")
        )
    )

    # ---------------------------------------------------------
    # Calculate correlations for nearby pairs
    # ---------------------------------------------------------

    spatial_correlations = []

    print("\n📊 Spatial Correlation Results:")

    # Evaluate up to 10 nearby pairs
    for sensor1_id, sensor2_id, distance in nearby_pairs[:10]:

        sensor1_data = (
            hourly_patterns
            .filter(
                F.col(sensor_col) == sensor1_id
            )
            .select(
                "hour_of_day",
                F.col("hourly_avg_value").alias("sensor1_value")
            )
        )

        sensor2_data = (
            hourly_patterns
            .filter(
                F.col(sensor_col) == sensor2_id
            )
            .select(
                "hour_of_day",
                F.col("hourly_avg_value").alias("sensor2_value")
            )
        )

        paired_data = (
            sensor1_data
            .join(
                sensor2_data,
                on="hour_of_day",
                how="inner"
            )
        )

        matching_hours = paired_data.count()

        if matching_hours >= 2:

            correlation = paired_data.stat.corr(
                "sensor1_value",
                "sensor2_value"
            )

            if correlation is not None:

                spatial_correlations.append(
                    {
                        "sensor_1": sensor1_id,
                        "sensor_2": sensor2_id,
                        "distance_km": distance,
                        "correlation": correlation,
                        "matching_hours": matching_hours
                    }
                )

                print(
                    f"   {sensor1_id} ↔ {sensor2_id} | "
                    f"Distance: {distance:.2f} km | "
                    f"Correlation: {correlation:.3f} | "
                    f"Matching hours: {matching_hours}"
                )

    # ---------------------------------------------------------
    # Handle no-results case
    # ---------------------------------------------------------

    if not spatial_correlations:

        print(
            "   No sampled nearby sensor pairs had enough "
            "matching hour-of-day observations for correlation."
        )

    else:

        avg_correlation = (
            builtins.sum(
                result["correlation"]
                for result in spatial_correlations
            )
            / len(spatial_correlations)
        )

        print(
            f"\n📈 Average spatial correlation: "
            f"{avg_correlation:.3f}"
        )

    return spatial_correlations


# ---------------------------------------------------------
# Run spatial analysis on traffic sensors
# ---------------------------------------------------------

spatial_correlations = analyze_spatial_correlations(
    datasets["traffic"],
    "vehicle_count",
    sensor_col="sensor_id",
    lat_col="location_lat",
    lon_col="location_lon",
    max_distance_km=2.0,
    sample_size=100
)


🗺️ SPATIAL CORRELATION ANALYSIS

🗺️ Spatial Correlation Analysis: vehicle_count
----------------------------------------
📍 Analyzing a sample of 100 sensors (from the full sensor population)
🔍 Found 1174 sampled sensor pairs within 2.0 km

📊 Spatial Correlation Results:
   TRF-0095 ↔ TRF-1413 | Distance: 1.13 km | Correlation: 0.003 | Matching hours: 12
   TRF-0095 ↔ TRF-1625 | Distance: 1.03 km | Correlation: -0.248 | Matching hours: 12
   TRF-0095 ↔ TRF-1023 | Distance: 1.77 km | Correlation: 0.682 | Matching hours: 12
   TRF-0095 ↔ TRF-0640 | Distance: 1.96 km | Correlation: 0.143 | Matching hours: 12
   TRF-0095 ↔ TRF-0831 | Distance: 1.88 km | Correlation: 0.526 | Matching hours: 12
   TRF-0095 ↔ TRF-1607 | Distance: 0.99 km | Correlation: 0.644 | Matching hours: 12

📈 Average spatial correlation: 0.292


## SECTION 3: FEATURE ENGINEERING

### TODO 3.1: Lag Features Creation

**Task:** Create lagged features for predictive modeling.  
**Concepts:** Time series forecasting, autoregression, feature lags.


In [19]:
print("\n" + "=" * 60)
print("⚙️ SECTION 3: FEATURE ENGINEERING")
print("=" * 60)

def create_lag_features(
    df,
    value_columns,
    lag_periods=[1, 6, 12, 24],
    time_col="timestamp",
    sensor_col=None
):
    """Create lag, difference, and percentage-change features."""
    print("\n🔄 Creating Lag Features")
    print("-" * 25)

    result_df = df
    window_spec = (
        Window.partitionBy(sensor_col).orderBy(time_col)
        if sensor_col
        else Window.orderBy(time_col)
    )

    for col_name in value_columns:
        if col_name not in df.columns:
            continue

        print(f"   Creating lags for {col_name}...")
        for lag_period in lag_periods:
            lag_col_name = f"{col_name}_lag_{lag_period}"
            result_df = result_df.withColumn(
                lag_col_name,
                F.lag(col_name, lag_period).over(window_spec)
            )

            non_null_lags = result_df.filter(
                F.col(lag_col_name).isNotNull()
            ).count()
            total_records = result_df.count()
            availability_pct = (
                non_null_lags / total_records * 100 if total_records else 0
            )
            print(
                f"      {lag_col_name}: "
                f"{availability_pct:.1f}% availability"
            )

        previous_value = F.lag(col_name, 1).over(window_spec)

        result_df = result_df.withColumn(
            f"{col_name}_diff_1",
            F.col(col_name) - previous_value
        )

        result_df = result_df.withColumn(
            f"{col_name}_pct_change_1",
            F.when(
                previous_value.isNotNull() & (previous_value != 0),
                ((F.col(col_name) - previous_value) / previous_value) * 100
            )
        )

    print(f"✅ Lag features created for {len(value_columns)} columns")
    return result_df


traffic_with_lags = create_lag_features(
    datasets["traffic"],
    ["vehicle_count", "avg_speed"],
    lag_periods=[1, 2, 3, 6],
    sensor_col="sensor_id"
)

print("\n📊 Sample Lag Features:")
lag_cols = [
    "timestamp", "sensor_id", "vehicle_count",
    "vehicle_count_lag_1", "vehicle_count_lag_6",
    "vehicle_count_diff_1", "vehicle_count_pct_change_1"
]
traffic_with_lags.select(
    [col for col in lag_cols if col in traffic_with_lags.columns]
).show(10)


⚙️ SECTION 3: FEATURE ENGINEERING

🔄 Creating Lag Features
-------------------------
   Creating lags for vehicle_count...
      vehicle_count_lag_1: 91.7% availability
      vehicle_count_lag_2: 83.3% availability
      vehicle_count_lag_3: 75.0% availability
      vehicle_count_lag_6: 50.0% availability
   Creating lags for avg_speed...
      avg_speed_lag_1: 91.7% availability
      avg_speed_lag_2: 83.3% availability
      avg_speed_lag_3: 75.0% availability
      avg_speed_lag_6: 50.0% availability
✅ Lag features created for 2 columns

📊 Sample Lag Features:
+-------------------+---------+-------------+-------------------+-------------------+--------------------+--------------------------+
|          timestamp|sensor_id|vehicle_count|vehicle_count_lag_1|vehicle_count_lag_6|vehicle_count_diff_1|vehicle_count_pct_change_1|
+-------------------+---------+-------------+-------------------+-------------------+--------------------+--------------------------+
|2025-01-01 00:00:00| TRF-0

### TODO 3.2: Rolling Statistics Features

In [20]:
# TODO 3.2: Rolling Statistics Features

def create_rolling_features(
    df,
    value_columns,
    windows=[3, 6, 12],
    time_col="timestamp",
    sensor_col=None
):
    """
    Calculate rolling statistics over previous sensor observations.

    Important:
    These windows represent numbers of observations, not fixed hours,
    because individual sensors do not report continuously in this dataset.
    """

    print("\n📊 Creating Rolling Statistics Features")
    print("-" * 35)

    result_df = df

    base_window = (
        Window.partitionBy(sensor_col).orderBy(time_col)
        if sensor_col
        else Window.orderBy(time_col)
    )

    for col_name in value_columns:

        if col_name not in df.columns:
            print(f"   ⚠️ Skipping missing column: {col_name}")
            continue

        print(f"   Creating rolling features for {col_name}...")

        for window_size in windows:

            # Include current row plus previous observations
            window_spec = base_window.rowsBetween(
                -window_size + 1,
                0
            )

            mean_col = f"{col_name}_rolling_mean_{window_size}"
            std_col = f"{col_name}_rolling_std_{window_size}"
            min_col = f"{col_name}_rolling_min_{window_size}"
            max_col = f"{col_name}_rolling_max_{window_size}"
            range_col = f"{col_name}_rolling_range_{window_size}"
            position_col = f"{col_name}_position_in_window_{window_size}"

            result_df = (
                result_df
                .withColumn(
                    mean_col,
                    F.avg(F.col(col_name)).over(window_spec)
                )
                .withColumn(
                    std_col,
                    F.stddev(F.col(col_name)).over(window_spec)
                )
                .withColumn(
                    min_col,
                    F.min(F.col(col_name)).over(window_spec)
                )
                .withColumn(
                    max_col,
                    F.max(F.col(col_name)).over(window_spec)
                )
                .withColumn(
                    range_col,
                    F.col(max_col) - F.col(min_col)
                )
                .withColumn(
                    position_col,
                    F.when(
                        F.col(range_col) > 0,
                        (
                            F.col(col_name) - F.col(min_col)
                        ) / F.col(range_col)
                    ).otherwise(F.lit(0.5))
                )
            )

            print(
                f"      Window {window_size}: "
                "mean, std, min, max, range, position"
            )

    print(
        f"✅ Rolling features created for "
        f"{len(value_columns)} columns"
    )

    return result_df


traffic_with_rolling = create_rolling_features(
    traffic_with_lags,
    ["vehicle_count", "avg_speed"],
    windows=[3, 6, 12],
    sensor_col="sensor_id"
)

print("\n📊 Sample Rolling Features:")

rolling_cols = [
    "timestamp",
    "sensor_id",
    "vehicle_count",
    "vehicle_count_rolling_mean_3",
    "vehicle_count_rolling_std_3",
    "vehicle_count_rolling_mean_6",
    "vehicle_count_rolling_std_6",
    "vehicle_count_position_in_window_6"
]

traffic_with_rolling.select(
    [
        col
        for col in rolling_cols
        if col in traffic_with_rolling.columns
    ]
).show(10, truncate=False)


📊 Creating Rolling Statistics Features
-----------------------------------
   Creating rolling features for vehicle_count...
      Window 3: mean, std, min, max, range, position
      Window 6: mean, std, min, max, range, position
      Window 12: mean, std, min, max, range, position
   Creating rolling features for avg_speed...
      Window 3: mean, std, min, max, range, position
      Window 6: mean, std, min, max, range, position
      Window 12: mean, std, min, max, range, position
✅ Rolling features created for 2 columns

📊 Sample Rolling Features:
+-------------------+---------+-------------+----------------------------+---------------------------+----------------------------+---------------------------+----------------------------------+
|timestamp          |sensor_id|vehicle_count|vehicle_count_rolling_mean_3|vehicle_count_rolling_std_3|vehicle_count_rolling_mean_6|vehicle_count_rolling_std_6|vehicle_count_position_in_window_6|
+-------------------+---------+-------------+----

### TODO 3.3: Interaction Features

**Task:** Engineer interaction features between related measurements.


In [21]:
def create_interaction_features(df, sensor_type):
    """Create domain-specific interaction features."""
    print(f"\n🔗 Creating Interaction Features: {sensor_type}")
    print("-" * 40)

    result_df = df
    interactions_created = []

    if sensor_type == "traffic":
        if {"vehicle_count", "avg_speed"}.issubset(df.columns):
            result_df = (
                result_df
                .withColumn(
                    "traffic_flow",
                    F.col("vehicle_count") * F.col("avg_speed")
                )
                .withColumn(
                    "congestion_indicator",
                    F.col("vehicle_count") / (F.col("avg_speed") + 1)
                )
                .withColumn(
                    "speed_efficiency",
                    F.when(
                        F.col("vehicle_count") > 0,
                        F.col("avg_speed") / F.sqrt(F.col("vehicle_count"))
                    ).otherwise(F.col("avg_speed"))
                )
            )
            interactions_created.extend([
                "traffic_flow", "congestion_indicator", "speed_efficiency"
            ])

    elif sensor_type == "air_quality":
        if {"pm25", "no2"}.issubset(df.columns):
            result_df = result_df.withColumn(
                "combined_pollution",
                F.col("pm25") * 0.6 + F.col("no2") * 0.4
            )
            interactions_created.append("combined_pollution")

        # if {"pm25", "temperature"}.issubset(df.columns):
        #     result_df = result_df.withColumn(
        #         "temp_adjusted_pm25",
        #         F.when(
        #             (F.col("temperature") + 10) != 0,
        #             F.col("pm25") / (F.col("temperature") + 10)
        #         )
        #     )
        #     interactions_created.append("temp_adjusted_pm25")

    elif sensor_type == "weather":
        if {"temperature", "humidity"}.issubset(df.columns):
            result_df = result_df.withColumn(
                "heat_index",
                F.col("temperature")
                + 0.5 * F.col("humidity") / 100
                * (F.col("temperature") - 14)
            )
            interactions_created.append("heat_index")

    elif sensor_type == "energy":
        if {"power_consumption", "voltage", "current"}.issubset(df.columns):
            denominator = F.col("voltage") * F.col("current") / 1000
            result_df = result_df.withColumn(
                "calculated_power_factor",
                F.when(
                    denominator != 0,
                    F.col("power_consumption") / denominator
                )
            )
            interactions_created.append("calculated_power_factor")

        if {"power_consumption", "building_type"}.issubset(df.columns):
            avg_by_building = df.groupBy("building_type").agg(
                F.avg("power_consumption").alias("avg_building_consumption")
            )
            result_df = result_df.join(
                avg_by_building, "building_type", "left"
            ).withColumn(
                "building_efficiency",
                F.when(
                    F.col("avg_building_consumption") != 0,
                    F.col("power_consumption") /
                    F.col("avg_building_consumption")
                )
            )
            interactions_created.append("building_efficiency")

    elif sensor_type == "occupancy":
        if {"available_rooms", "occupied_rooms"}.issubset(df.columns):
            total_rooms = F.col("available_rooms") + F.col("occupied_rooms")
            result_df = (
                result_df
                .withColumn("total_rooms", total_rooms)
                .withColumn(
                    "occupancy_rate",
                    F.when(
                        total_rooms > 0,
                        F.col("occupied_rooms") / total_rooms
                    )
                )
            )
            interactions_created.extend(["total_rooms", "occupancy_rate"])

    elif sensor_type == "fiscal":
        if {"expense", "revenue"}.issubset(df.columns):
            result_df = result_df.withColumn(
                "net_profit",
                F.col("revenue") - F.col("expense")
            )
            interactions_created.append("net_profit")

    print(f"🔗 Interaction features created: {interactions_created}")
    return result_df


feature_datasets = {}

for name, df in datasets.items():
    if df is None or name == "zones":
        continue

    try:
        # Keep the full traffic feature pipeline together.
        source_df = traffic_with_rolling if name == "traffic" else df
        feature_datasets[name] = create_interaction_features(source_df, name)
        print(f"✅ Interaction features created for {name}")
    except Exception as e:
        print(f"❌ Error creating interactions for {name}: {str(e)}")


🔗 Creating Interaction Features: traffic
----------------------------------------
🔗 Interaction features created: ['traffic_flow', 'congestion_indicator', 'speed_efficiency']
✅ Interaction features created for traffic

🔗 Creating Interaction Features: air_quality
----------------------------------------
🔗 Interaction features created: ['combined_pollution']
✅ Interaction features created for air_quality

🔗 Creating Interaction Features: weather
----------------------------------------
🔗 Interaction features created: ['heat_index']
✅ Interaction features created for weather

🔗 Creating Interaction Features: energy
----------------------------------------
🔗 Interaction features created: ['calculated_power_factor', 'building_efficiency']
✅ Interaction features created for energy

🔗 Creating Interaction Features: occupancy
----------------------------------------
🔗 Interaction features created: ['total_rooms', 'occupancy_rate']
✅ Interaction features created for occupancy

🔗 Creating Inte

## SECTION 4: TREND ANALYSIS

### TODO 4.1: Trend Detection and Quantification

**Task:** Identify and quantify trends in sensor data.  
**Concepts:** Trend analysis, rate of change, long-term vs. short-term patterns.


In [22]:
# TODO 4.1: Trend Detection and Quantification

print("\n" + "=" * 60)
print("📈 SECTION 4: TREND ANALYSIS")
print("=" * 60)


def detect_and_quantify_trends(
    df,
    value_col,
    time_col="timestamp",
    sensor_col=None,
    trend_period=6
):
    """
    Identify and quantify trends in sensor data.

    trend_period represents the number of previous observations,
    not a fixed number of hours.
    """

    print(f"\n📈 Trend Detection: {value_col}")
    print("-" * 30)

    print(
        f"   Using {trend_period} previous observations "
        "for trend comparison"
    )

    # ---------------------------------------------------------
    # Create numeric time fields
    # ---------------------------------------------------------

    min_timestamp = df.agg(
        F.min(time_col).alias("min_timestamp")
    ).first()["min_timestamp"]

    df_with_time_numeric = (
        df
        .withColumn(
            "time_numeric",
            F.unix_timestamp(time_col)
        )
        .withColumn(
            "day_numeric",
            F.datediff(
                F.to_date(F.col(time_col)),
                F.lit(min_timestamp.date())
            )
        )
    )

    # ---------------------------------------------------------
    # Create appropriate window
    # ---------------------------------------------------------

    if sensor_col:

        print("   Calculating per-sensor trends...")

        window_spec = (
            Window
            .partitionBy(sensor_col)
            .orderBy(time_col)
        )

    else:

        print("   Calculating global trends...")

        window_spec = (
            Window
            .orderBy(time_col)
        )

    # Previous value from trend_period observations ago
    previous_value = F.lag(
        value_col,
        trend_period
    ).over(window_spec)

    # Rolling window used to estimate variability
    trend_window = (
        window_spec
        .rowsBetween(
            -trend_period,
            0
        )
    )

    # ---------------------------------------------------------
    # Create trend features
    # ---------------------------------------------------------

    df_with_trends = (
        df_with_time_numeric

        .withColumn(
            f"{value_col}_trend_reference",
            previous_value
        )

        .withColumn(
            f"{value_col}_trend_direction",

            F.when(
                previous_value.isNull(),
                F.lit(0)
            )

            .when(
                F.col(value_col) > previous_value,
                F.lit(1)
            )

            .when(
                F.col(value_col) < previous_value,
                F.lit(-1)
            )

            .otherwise(
                F.lit(0)
            )
        )

        .withColumn(
            f"{value_col}_trend_strength",

            F.when(
                previous_value.isNotNull(),

                F.abs(
                    F.col(value_col)
                    - previous_value
                )
                /
                (
                    F.stddev(
                        value_col
                    ).over(trend_window)
                    + F.lit(0.001)
                )
            )
        )
    )

    # ---------------------------------------------------------
    # Summarize trends
    # ---------------------------------------------------------

    if sensor_col:

        trend_summary = (
            df_with_trends
            .groupBy(sensor_col)
            .agg(

                F.avg(
                    f"{value_col}_trend_direction"
                ).alias(
                    "avg_trend_direction"
                ),

                F.avg(
                    f"{value_col}_trend_strength"
                ).alias(
                    "avg_trend_strength"
                )
            )
        )

        print("\n   📊 Trend Summary by Sensor:")

        trend_summary.show(
            10,
            truncate=False
        )

    else:

        overall_trend = (
            df_with_trends
            .agg(

                F.avg(
                    f"{value_col}_trend_direction"
                ).alias(
                    "overall_trend_direction"
                ),

                F.avg(
                    f"{value_col}_trend_strength"
                ).alias(
                    "overall_trend_strength"
                )
            )
            .first()
        )

        print(
            f"   📊 Overall Trend Direction: "
            f"{overall_trend['overall_trend_direction']:.3f}"
        )

        print(
            f"   📊 Overall Trend Strength: "
            f"{overall_trend['overall_trend_strength']:.3f}"
        )

    return df_with_trends


# ---------------------------------------------------------
# Traffic Trend Analysis
# ---------------------------------------------------------

print("🚗 Analyzing trends in traffic data...")

traffic_trends = detect_and_quantify_trends(
    feature_datasets["traffic"],
    "vehicle_count",
    sensor_col="sensor_id",
    trend_period=6
)

# Keep trend features in the final traffic feature dataset
feature_datasets["traffic"] = traffic_trends


# ---------------------------------------------------------
# Display sample trend results
# ---------------------------------------------------------

print("\n📊 Sample Trend Analysis:")

trend_cols = [
    "timestamp",
    "sensor_id",
    "vehicle_count",
    "vehicle_count_trend_reference",
    "vehicle_count_trend_direction",
    "vehicle_count_trend_strength"
]

traffic_trends.select(
    [
        col
        for col in trend_cols
        if col in traffic_trends.columns
    ]
).show(
    15,
    truncate=False
)


📈 SECTION 4: TREND ANALYSIS
🚗 Analyzing trends in traffic data...

📈 Trend Detection: vehicle_count
------------------------------
   Using 6 previous observations for trend comparison
   Calculating per-sensor trends...

   📊 Trend Summary by Sensor:
+---------+-------------------+------------------+
|sensor_id|avg_trend_direction|avg_trend_strength|
+---------+-------------------+------------------+
|TRF-0001 |0.0                |0.8452760965136913|
|TRF-0002 |0.0                |1.000581935806924 |
|TRF-0003 |0.16666666666666666|1.0014819202747554|
|TRF-0004 |0.0                |1.5002022950445495|
|TRF-0005 |0.16666666666666666|1.7687900767515412|
|TRF-0006 |0.0                |1.064359874983202 |
|TRF-0007 |0.16666666666666666|0.8609716415380658|
|TRF-0008 |0.3333333333333333 |1.3181895430259989|
|TRF-0009 |0.0                |1.5071814541109203|
|TRF-0010 |-0.3333333333333333|1.0007585663984802|
+---------+-------------------+------------------+
only showing top 10 rows

📊 Sampl

## DAY 3 DELIVERABLES & VALIDATION

In [25]:
print("\n" + "=" * 60)
print("📋 DAY 3 COMPLETION CHECKLIST")
print("=" * 60)

import builtins

def validate_day3_completion():
    """Validate that the Day 3 objectives have been met."""
    checklist = {
        "temporal_patterns_analyzed": bool(temporal_results),
        "pattern_anomalies_detected": (
            "traffic_with_anomalies" in globals()
            and traffic_with_anomalies is not None
            and "vehicle_count_anomaly_weekly" in traffic_with_anomalies.columns
        ),
        "correlation_analysis_completed": (
            "correlations" in globals()
            and correlations is not None
        ),
        "spatial_correlations_analyzed": (
            "spatial_correlations" in globals()
            and len(spatial_correlations) > 0
        ),
        "lag_features_created": (
            "traffic_with_lags" in globals()
            and traffic_with_lags is not None
        ),
        "rolling_features_created": (
            "traffic_with_rolling" in globals()
            and traffic_with_rolling is not None
        ),
        "interaction_features_engineered": bool(feature_datasets),
        "trend_analysis_completed": (
            "traffic_trends" in globals()
            and traffic_trends is not None
        ),
        "feature_pipeline_documented": bool(feature_datasets),
    }

    print("✅ COMPLETION STATUS:")
    for item, status in checklist.items():
        icon = "✅" if status else "❌"
        print(f"   {icon} {item.replace('_', ' ').title()}")

    completion_rate = (
        builtins.sum(checklist.values()) / len(checklist) * 100
    )
    print(f"\n📊 Overall Completion: {completion_rate:.1f}%")

    if completion_rate == 100:
        print("🎉 Day 3 objectives completed.")
    else:
        print("📝 Review incomplete items before proceeding to Day 4.")

    return checklist


completion_status = validate_day3_completion()


📋 DAY 3 COMPLETION CHECKLIST
✅ COMPLETION STATUS:
   ✅ Temporal Patterns Analyzed
   ✅ Pattern Anomalies Detected
   ✅ Correlation Analysis Completed
   ✅ Spatial Correlations Analyzed
   ✅ Lag Features Created
   ✅ Rolling Features Created
   ✅ Interaction Features Engineered
   ✅ Trend Analysis Completed
   ✅ Feature Pipeline Documented

📊 Overall Completion: 100.0%
🎉 Day 3 objectives completed.


## SAVE ENGINEERED FEATURES FOR DAY 4

In [26]:
# Save Day 3 feature-engineered datasets

features_data_dir = "../data/features"

os.makedirs(features_data_dir, exist_ok=True)

print("\n💾 Saving Day 3 Feature-Engineered Datasets")
print("-" * 50)

for name, df in feature_datasets.items():

    output_path = f"{features_data_dir}/{name}_features.parquet"

    print(f"   Saving {name}...")

    df.write.mode("overwrite").parquet(output_path)

    print(f"   ✅ Saved: {output_path}")

print("\n🎉 All Day 3 feature datasets saved successfully.")


💾 Saving Day 3 Feature-Engineered Datasets
--------------------------------------------------
   Saving traffic...


   ✅ Saved: ../data/features/traffic_features.parquet
   Saving air_quality...
   ✅ Saved: ../data/features/air_quality_features.parquet
   Saving weather...
   ✅ Saved: ../data/features/weather_features.parquet
   Saving energy...
   ✅ Saved: ../data/features/energy_features.parquet
   Saving occupancy...
   ✅ Saved: ../data/features/occupancy_features.parquet
   Saving fiscal...
   ✅ Saved: ../data/features/fiscal_features.parquet

🎉 All Day 3 feature datasets saved successfully.


## Day 3 Findings & Notes

Complete this section after reviewing the outputs from Sections 2–4.

Items to document:
- strongest cross-sensor correlations and what they may indicate;
- spatial-correlation observations;
- lag and rolling features created for traffic;
- interaction features created for each dataset;
- traffic trend findings;
- the 20 weekly traffic anomalies identified in Section 1;
- any data-quality limitations that affect interpretation, including the previously identified future-dated weather records and overlapping zone mappings.


## What's Next?

**Day 4 Preview: Advanced Analytics & Anomaly Detection**

Day 4 will use the engineered features produced here for advanced anomaly detection and predictive modeling.


## Day 3 Findings, Feature Engineering Decisions & Analysis Notes

### Key Findings

- **Traffic patterns:** Vehicle volume shows clear commute-related peaks, particularly around 7–8 AM and 4–6 PM. The highest average vehicle count occurred around 5 PM. Average traffic speed decreased during the higher-volume commute periods.
- **Air quality:** PM2.5 levels remained relatively stable across the analyzed time periods, with little difference between weekday and weekend averages.
- **Energy usage:** Energy consumption showed noticeable peaks around 6–8 AM and 5–7 PM.
- **Occupancy:** Average occupied rooms were higher on weekends than weekdays. Occupancy increased into the summer months before declining later in the year.
- **Fiscal activity:** Revenue was higher on weekends than weekdays and followed a seasonal pattern similar to occupancy.
- **Cross-dataset correlation:** Weather temperature and humidity had the strongest observed relationship, with a strong negative correlation of approximately -0.967. Traffic volume and average traffic speed had a moderate negative correlation of approximately -0.586.
- **Spatial traffic correlation:** Nearby traffic sensors showed mixed relationships. The analyzed sensor pairs produced an average spatial correlation of approximately 0.292, indicating a weak positive relationship overall. Proximity alone does not appear to strongly determine similar traffic behavior.

### Feature Engineering Decisions

- Lag features were created using previous **sensor observations**, rather than fixed time intervals.
- Rolling statistics use 3-, 6-, and 12-observation windows.
- Trend analysis compares the current traffic reading with the reading from 6 observations earlier.
- These observation-based windows were selected because the generated traffic dataset contains approximately 12 readings per individual sensor. Larger starter windows such as 24 and 48 observations would provide limited additional information for this dataset.
- Spatial correlation was evaluated using **hour-of-day traffic patterns** instead of exact timestamp matches because individual sensors report sparsely and generally do not share matching timestamps.
- Interaction features were created for traffic, air quality, weather, energy, occupancy, and fiscal datasets to support later analysis and convention-focused use cases.

### Data Quality & Interpretation Notes

- Statistical outliers identified during Day 2 and used in subsequent analysis do not automatically represent invalid data. Domain validation did not identify major impossible values.
- Weather records extend beyond the current analysis date. These future-dated timestamps should be reviewed before the data is used for production reporting or a convention application.
- Earlier zone mapping produced 36,079 mapped traffic records from 36,000 original traffic records, indicating that some coordinates may match overlapping zone boundaries. This remains a data-quality item for the team to review.
- The temporal analysis performed in Day 3 identifies hourly, daily, weekly, and monthly patterns. It should not be described as a formal statistical seasonal decomposition into trend, seasonal, and residual components.

### Convention Project Considerations

Day 3 provides analytical features that may help determine what information is useful for the SparkCity convention project. Not every engineered feature needs to appear in the final dashboard or Java application.

Potentially useful convention-focused information includes:

- traffic volume, congestion, and travel patterns;
- occupancy and capacity trends;
- weather conditions;
- air-quality conditions;
- energy usage;
- fiscal/economic activity; and
- location or zone-based relationships.

The next phase should focus on selecting the features that answer actual convention business questions and defining the clean, predictable data that will be provided to the Java team.